<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSEaxoapR7GyB6FC7xp4aM-SrjhxTWTdFCmKAX3TvvFvQ&s=10" alt="Tecnológico de Costa Rica" width="160">

# 🌎 WALC 2026
### Ciencia de Datos · Introducción al Aprendizaje Automático
**Día 4 (Jueves) — No supervisado — Agrupación (K-means) y asociación (Apriori)**

**Instituto Tecnológico de Costa Rica (TEC)** · Instructor: **Efrén Jiménez Delgado**
· 📧 efjimenez@itcr.ac.cr · 🔗 [LinkedIn](https://www.linkedin.com/in/ejimenezdelgado/)

# 🎯 Agrupamiento: teoría de K-means, jerárquico y DBSCAN, y K-means desde cero

## 🧵 El hilo conductor (metodología CRISP-DM)

Este cuaderno sigue las seis fases de **CRISP-DM** [1] [2]: cada título de fase marca dónde estamos en el proceso.

| Fase | En este cuaderno |
|---|---|
| 🎯 Comprensión del negocio | Entender qué es agrupar sin etiquetas y cómo funciona K-means. |
| 🔍 Comprensión de los datos | Datos sintéticos con grupos conocidos para poder medir qué tan bien agrupamos. |
| 🧹 Preparación de los datos | Estandarización: K-means depende de la escala. |
| 🤖 Modelado | K-means desde cero, comparado con scikit-learn; alternativas jerárquica y DBSCAN. |
| 📊 Evaluación | Codo, silueta y concordancia (Rand ajustado) con los grupos reales. |
| 🚀 Despliegue y conclusiones | Buenas prácticas para llevarlo a un caso real. |

> 🎯 **Analogía:** acomodar a los invitados de una fiesta en mesas: cada quien se sienta en la mesa cuyo "centro" le queda más cerca, y las mesas se reacomodan hasta que nadie se mueve.

### 🗺️ Mapa del curso (cronograma Track 5 · Ciencia de datos)
| Día | Jornada | Teoría y laboratorio |
|---|---|---|
| 1 | Lunes | Predicción: regresión lineal y series de tiempo |
| 2 | Martes | Clasificación 1: árboles de decisión, bosques aleatorios y KNN |
| 3 | Miércoles | Clasificación 2: SVM, regresión logística, Naive Bayes y redes neuronales |
| 👉 **4** | Jueves | **No supervisado: agrupación y asociación** |
| 5 | Viernes | Ciencia de datos con IA generativa, ética y buenas prácticas |

## 1. Entendimiento del problema (fase 1 de CRISP-DM)

En **aprendizaje no supervisado** no existe una columna "respuesta": se buscan estructuras en los datos. El **agrupamiento** (*clustering*) reparte las observaciones en grupos de modo que las de un mismo grupo se parezcan entre sí más que a las de otros [8]. Ejemplos del curso: segmentar clientes, agrupar delitos por zona.

![Datos sin etiquetas con tres nubes de puntos naturales](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch11/images/11_01.png)
*Figura: Datos sin etiquetas con tres nubes de puntos naturales. [11]*

## 2. Teoría de K-means

K-means busca *k* centroides $\mu_1,\dots,\mu_k$ que minimicen la **suma de cuadrados dentro de los grupos** (inercia o SSE):

$$ J=\sum_{j=1}^{k}\sum_{x_i\in C_j}\lVert x_i-\mu_j\rVert^{2} $$

El **algoritmo de Lloyd** [4] [3] alterna dos pasos hasta que nada cambia: (1) **asignar** cada punto al centroide más cercano; (2) **actualizar** cada centroide como la media de sus puntos. Cada iteración no aumenta $J$, pero solo garantiza un **mínimo local**, por eso importa la inicialización: **k-means++** elige centroides iniciales lejanos entre sí [5].

![K-means con k = 3: grupos y centroides (estrellas rojas)](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch11/images/11_02.png)
*Figura: K-means con k = 3: grupos y centroides (estrellas rojas). [11]*

**Supuestos y límites:** grupos de forma aproximadamente esférica y tamaño parecido; sensible a la escala de las variables y a los valores atípicos; hay que fijar *k* de antemano [8].

## 3. Preparación y modelado: K-means desde cero

Se programa el algoritmo de Lloyd con NumPy y se compara con `KMeans` de scikit-learn [10] sobre datos sintéticos con tres grupos conocidos.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

X, y_real = make_blobs(n_samples=300, centers=3, cluster_std=1.0, random_state=7)

def kmeans_desde_cero(X, k, iteraciones=100, semilla=0):
    rng = np.random.default_rng(semilla)
    centros = X[rng.choice(len(X), k, replace=False)]
    for _ in range(iteraciones):
        dist = np.linalg.norm(X[:, None, :] - centros[None, :, :], axis=2)   # distancia de cada punto a cada centro
        etiquetas = dist.argmin(axis=1)                                      # paso 1: asignar
        nuevos = np.array([X[etiquetas == j].mean(axis=0) if np.any(etiquetas == j) else centros[j] for j in range(k)])  # paso 2: actualizar
        if np.allclose(nuevos, centros): break
        centros = nuevos
    sse = ((X - centros[etiquetas]) ** 2).sum()
    return etiquetas, centros, sse

# varias semillas y nos quedamos con la de menor SSE (lo que hace n_init)
mejor = min((kmeans_desde_cero(X, 3, semilla=s) for s in range(10)), key=lambda r: r[2])
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X)
print(f'SSE desde cero: {mejor[2]:.1f} | SSE sklearn: {km.inertia_:.1f}')
print('Concordancia entre ambos (Rand ajustado):', round(adjusted_rand_score(mejor[0], km.labels_), 3))
print('Concordancia con los grupos reales:', round(adjusted_rand_score(y_real, mejor[0]), 3))
plt.scatter(X[:, 0], X[:, 1], c=mejor[0], s=15); plt.scatter(*mejor[1].T, c='red', marker='X', s=150); plt.title('K-means desde cero'); plt.show()

## 4. Evaluación: ¿cuántos grupos?

- **Método del codo:** se grafica la inercia contra *k* y se busca el punto donde deja de bajar bruscamente [6].
- **Coeficiente de silueta:** para cada punto compara su distancia media a su propio grupo ($a$) con la del grupo vecino más cercano ($b$): $s=\dfrac{b-a}{\max(a,b)}$, entre −1 y 1; cuanto más cerca de 1, mejor [7].

![Método del codo: la distorsión deja de caer bruscamente en k = 3](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch11/images/11_03.png)
*Figura: Método del codo: la distorsión deja de caer bruscamente en k = 3. [11]*

![Gráfico de silueta de un agrupamiento de buena calidad (k = 3)](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch11/images/11_04.png)
*Figura: Gráfico de silueta de un agrupamiento de buena calidad (k = 3). [11]*

![Gráfico de silueta de un agrupamiento deficiente (k = 2)](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch11/images/11_06.png)
*Figura: Gráfico de silueta de un agrupamiento deficiente (k = 2). [11]*

In [ ]:
ks = range(2, 9)
inercia = [KMeans(n_clusters=k, n_init=10, random_state=0).fit(X).inertia_ for k in ks]
silueta = [silhouette_score(X, KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)) for k in ks]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(ks, inercia, 'o-'); ax[0].set_title('Codo (inercia)'); ax[0].set_xlabel('k')
ax[1].plot(ks, silueta, 'o-'); ax[1].set_title('Silueta'); ax[1].set_xlabel('k')
plt.show()
print('k con mayor silueta:', list(ks)[int(np.argmax(silueta))])

## 5. Cuando K-means no basta: jerárquico y DBSCAN

El agrupamiento **jerárquico aglomerativo** une los grupos más cercanos paso a paso y se resume en un **dendrograma**; el criterio de enlace (simple, completo, promedio, Ward) define qué significa "cercano" [9].

![Enlace simple y completo en agrupamiento jerárquico](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch11/images/11_07.png)
*Figura: Enlace simple y completo en agrupamiento jerárquico. [11]*

![Dendrograma: cómo se van uniendo los grupos](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch11/images/11_11.png)
*Figura: Dendrograma: cómo se van uniendo los grupos. [11]*

**DBSCAN** agrupa por **densidad**: un punto es *núcleo* si tiene al menos `min_samples` vecinos a distancia `eps`; los puntos sin vecindad densa son **ruido**. Encuentra grupos de forma arbitraria y no necesita fijar *k* [8].

![DBSCAN: puntos núcleo, de borde y ruido](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch11/images/11_13.png)
*Figura: DBSCAN: puntos núcleo, de borde y ruido. [11]*

![K-means vs. jerárquico en datos con forma de lunas](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch11/images/11_15.png)
*Figura: K-means vs. jerárquico en datos con forma de lunas. [11]*

In [ ]:
from sklearn.datasets import make_moons
from sklearn.cluster import DBSCAN, AgglomerativeClustering

Xm, ym = make_moons(n_samples=300, noise=0.06, random_state=0)
modelos = {
    'K-means (k=2)': KMeans(n_clusters=2, n_init=10, random_state=0),
    'Jerárquico, enlace simple': AgglomerativeClustering(n_clusters=2, linkage='single'),
    'DBSCAN (eps=0.2)': DBSCAN(eps=0.2, min_samples=5),
}
fig, ax = plt.subplots(1, 3, figsize=(12, 3.5))
for a, (nombre, m) in zip(ax, modelos.items()):
    et = m.fit_predict(Xm)
    a.scatter(Xm[:, 0], Xm[:, 1], c=et, s=10); a.set_title(f'{nombre}\nRand ajustado = {adjusted_rand_score(ym, et):.2f}')
plt.tight_layout(); plt.show()

### Escala de las variables

K-means usa distancias, de modo que una variable con valores grandes domina. Comprobamos el efecto: una variable irrelevante con escala enorme arruina el agrupamiento; al estandarizar mejora, aunque la variable de ruido sigue restando calidad.

In [ ]:
from sklearn.preprocessing import StandardScaler
rng = np.random.default_rng(0)
ruido = rng.normal(0, 1000, size=(len(X), 1))            # variable sin información, en miles
Xr = np.hstack([X, ruido])
sin = KMeans(3, n_init=10, random_state=0).fit_predict(Xr)
con = KMeans(3, n_init=10, random_state=0).fit_predict(StandardScaler().fit_transform(Xr))
print('Concordancia con los grupos reales SIN estandarizar:', round(adjusted_rand_score(y_real, sin), 3))
print('Concordancia con los grupos reales estandarizando:  ', round(adjusted_rand_score(y_real, con), 3))

## 6. Despliegue y buenas prácticas

- Estandarice (y transforme con logaritmo si hay asimetría) antes de agrupar.
- Pruebe varias semillas (`n_init`) y verifique la **estabilidad** de los grupos.
- Combine codo, silueta y, sobre todo, **criterio del negocio** para elegir *k*.
- Perfile cada grupo (medias por variable) y nómbrelo; un grupo que no se puede explicar no es accionable.

### ✍️ Ejercicios
1. Cambie `cluster_std` a 3. ¿Qué pasa con la silueta y con la concordancia con los grupos reales?
2. Agrupe `make_blobs` con tamaños muy distintos (`n_samples=[250, 30, 20]`). ¿Qué hace K-means? ¿Y DBSCAN?
3. En `make_moons`, pruebe `eps` = 0.1, 0.2 y 0.5. ¿Cómo cambia el número de grupos y de puntos ruido?

---
## ✅ Conclusiones y siguientes pasos

K-means es rápido y fácil de interpretar, pero supone grupos compactos y de tamaño parecido y exige elegir *k* y estandarizar [8]. Los datos con forma de lunas muestran el límite: K-means los parte mal, mientras que DBSCAN y el enlace simple los separan. La elección del método y de *k* debe validarse con silueta, estabilidad y criterio del negocio [7].

---
## 📚 Referencias

**[1]** Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., & Wirth, R. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*. SPSS Inc. https://www.the-modeling-agency.com/crisp-dm.pdf

**[2]** Wirth, R., & Hipp, J. (2000). CRISP-DM: Towards a standard process model for data mining. *Proceedings of the 4th International Conference on the Practical Applications of Knowledge Discovery and Data Mining*, 29–39.

**[3]** MacQueen, J. (1967). Some methods for classification and analysis of multivariate observations. *Proceedings of the Fifth Berkeley Symposium on Mathematical Statistics and Probability, 1*, 281–297.

**[4]** Lloyd, S. P. (1982). Least squares quantization in PCM. *IEEE Transactions on Information Theory, 28*(2), 129–137. https://doi.org/10.1109/TIT.1982.1056489

**[5]** Arthur, D., & Vassilvitskii, S. (2007). k-means++: The advantages of careful seeding. *Proceedings of the 18th Annual ACM-SIAM Symposium on Discrete Algorithms (SODA '07)*, 1027–1035.

**[6]** Thorndike, R. L. (1953). Who belongs in the family? *Psychometrika, 18*(4), 267–276. https://doi.org/10.1007/BF02289263

**[7]** Rousseeuw, P. J. (1987). Silhouettes: A graphical aid to the interpretation and validation of cluster analysis. *Journal of Computational and Applied Mathematics, 20*, 53–65. https://doi.org/10.1016/0377-0427(87)90125-7

**[8]** Jain, A. K. (2010). Data clustering: 50 years beyond K-means. *Pattern Recognition Letters, 31*(8), 651–666. https://doi.org/10.1016/j.patrec.2009.09.011

**[9]** Han, J., Kamber, M., & Pei, J. (2012). *Data Mining: Concepts and Techniques* (3.ª ed.). Morgan Kaufmann. https://doi.org/10.1016/C2009-0-61819-5

**[10]** Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., … Duchesnay, E. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830. https://jmlr.org/papers/v12/pedregosa11a.html

**[11]** Raschka, S., & Mirjalili, V. (2019). *Python Machine Learning* (3.ª ed.). Packt Publishing. (Figuras usadas desde https://github.com/rasbt/python-machine-learning-book-3rd-edition)